# TTSPort Testing Guide

This notebook tests the TTSPort implementation with real API calls.

**Prerequisites:**
- Set `ELEVENLABS_API_KEY` environment variable for ElevenLabs tests
- Set `OPENAI_API_KEY` environment variable for OpenAI tests

In [1]:
# Load environment variables
import os
from dotenv import load_dotenv
load_dotenv()

print(f"ELEVENLABS_API_KEY set: {bool(os.getenv('ELEVENLABS_API_KEY'))}")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")

ELEVENLABS_API_KEY set: True
OPENAI_API_KEY set: True


## 1. ElevenLabs TTS

In [2]:
from chatforge.adapters.tts import ElevenLabsTTSAdapter, ElevenLabsVoiceConfig
from chatforge.ports.tts import AudioFormat, AudioQuality

In [ ]:
# List available voices
async with ElevenLabsTTSAdapter() as tts:
    voices = await tts.list_voices()
    print(f"Found {len(voices)} voices:\n")
    for v in voices[:5]:  # Show first 5
        print(f"  - {v.name} ({v.voice_id})")

In [ ]:
# Synthesize with ElevenLabs (using default voice)
# Rachel voice ID - a common default voice
VOICE_ID = "21m00Tcm4TlvDq8ikWAM"

async with ElevenLabsTTSAdapter() as tts:
    config = ElevenLabsVoiceConfig(
        voice_id=VOICE_ID,
        stability=0.5,
        similarity_boost=0.75,
    )
    
    # Use STANDARD quality (HIGH requires Creator tier)
    result = await tts.synthesize(
        "Hello world this is elevanlabs speaking yo",
        config,
        output_format=AudioFormat.MP3,
        quality=AudioQuality.STANDARD,
    )
    
    print(f"Audio bytes: {len(result.audio_bytes)}")
    print(f"Format: {result.format}")
    print(f"Sample rate: {result.sample_rate}")
    print(f"Input characters: {result.input_characters}")
    
    # Save to file
    with open("elevenlabs_output.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("\nSaved to elevenlabs_output.mp3")

Audio bytes: 62320
Format: AudioFormat.MP3
Sample rate: 44100
Input characters: 73

Saved to elevenlabs_output.mp3


## 2. OpenAI TTS

In [6]:
from chatforge.adapters.tts import OpenAITTSAdapter, OpenAIVoiceConfig
from chatforge.ports.tts import AudioFormat, AudioQuality

In [7]:
# List available voices
async with OpenAITTSAdapter() as tts:
    voices = await tts.list_voices()
    print(f"Found {len(voices)} voices:\n")
    for v in voices:
        print(f"  - {v.name} ({v.voice_id})")

Found 11 voices:

  - Alloy (alloy)
  - Ash (ash)
  - Ballad (ballad)
  - Coral (coral)
  - Echo (echo)
  - Fable (fable)
  - Nova (nova)
  - Onyx (onyx)
  - Sage (sage)
  - Shimmer (shimmer)
  - Verse (verse)


In [8]:
# Synthesize with OpenAI
async with OpenAITTSAdapter() as tts:
    config = OpenAIVoiceConfig(
        voice_id="nova",
        style_prompt="Speak in a warm, friendly tone",
    )
    
    result = await tts.synthesize(
        "Hello world! This is a test of the OpenAI text to speech adapter.",
        config,
        output_format=AudioFormat.MP3,
        quality=AudioQuality.HIGH,
    )
    
    print(f"Audio bytes: {len(result.audio_bytes)}")
    print(f"Format: {result.format}")
    print(f"Sample rate: {result.sample_rate}")
    print(f"Input characters: {result.input_characters}")
    
    # Save to file
    with open("openai_output.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("\nSaved to openai_output.mp3")

Audio bytes: 82080
Format: AudioFormat.MP3
Sample rate: 24000
Input characters: 65

Saved to openai_output.mp3


## 3. Streaming Example

In [9]:
# Streaming with OpenAI
async with OpenAITTSAdapter() as tts:
    config = OpenAIVoiceConfig(voice_id="alloy")
    
    chunks = []
    async for chunk in tts.stream(
        "This is a streaming test. The audio is generated in chunks.",
        config,
    ):
        chunks.append(chunk)
        print(f"Received chunk: {len(chunk)} bytes")
    
    total_bytes = sum(len(c) for c in chunks)
    print(f"\nTotal: {len(chunks)} chunks, {total_bytes} bytes")
    
    # Save streamed audio
    with open("openai_streamed.mp3", "wb") as f:
        for chunk in chunks:
            f.write(chunk)
    print("Saved to openai_streamed.mp3")

Received chunk: 107 bytes
Received chunk: 1369 bytes
Received chunk: 1369 bytes
Received chunk: 1207 bytes
Received chunk: 1364 bytes
Received chunk: 1128 bytes
Received chunk: 349 bytes
Received chunk: 928 bytes
Received chunk: 1364 bytes
Received chunk: 758 bytes
Received chunk: 991 bytes
Received chunk: 1285 bytes
Received chunk: 474 bytes
Received chunk: 1199 bytes
Received chunk: 739 bytes
Received chunk: 188 bytes
Received chunk: 1364 bytes
Received chunk: 189 bytes
Received chunk: 722 bytes
Received chunk: 1364 bytes
Received chunk: 1369 bytes
Received chunk: 457 bytes
Received chunk: 544 bytes
Received chunk: 474 bytes
Received chunk: 42 bytes
Received chunk: 177 bytes
Received chunk: 101 bytes
Received chunk: 801 bytes
Received chunk: 225 bytes
Received chunk: 86 bytes
Received chunk: 50 bytes
Received chunk: 127 bytes
Received chunk: 166 bytes
Received chunk: 70 bytes
Received chunk: 210 bytes
Received chunk: 63 bytes
Received chunk: 724 bytes
Received chunk: 260 bytes
Receiv

## 4. Play Audio (Optional)

If you have IPython.display available, you can play the audio directly:

In [10]:
from IPython.display import Audio, display

# Play the OpenAI output
display(Audio("openai_output.mp3"))

## 5. Hexagonal Architecture Pattern

In proper hexagonal architecture, application code should depend on the **TTSPort interface**, not concrete adapters. This allows swapping providers without changing business logic.

In [12]:
from chatforge.ports.tts import TTSPort, VoiceConfig, AudioResult, AudioFormat, AudioQuality
from chatforge.adapters.tts import (
    ElevenLabsTTSAdapter, ElevenLabsVoiceConfig,
    OpenAITTSAdapter, OpenAIVoiceConfig,
)


async def generate_speech(tts: TTSPort, text: str, config: VoiceConfig) -> AudioResult:
    """
    Application code depends ONLY on TTSPort interface.
    Provider can be swapped without changing this function.
    """
    return await tts.synthesize(
        text, 
        config, 
        output_format=AudioFormat.MP3,
        quality=AudioQuality.STANDARD,
    )

In [13]:
# ElevenLabs via TTSPort interface
text = "This demonstrates the hexagonal architecture pattern with ElevenLabs."

async with ElevenLabsTTSAdapter() as adapter:
    config = ElevenLabsVoiceConfig(voice_id="21m00Tcm4TlvDq8ikWAM")
    result = await generate_speech(adapter, text, config)
    
    print(f"ElevenLabs: {len(result.audio_bytes)} bytes")
    
    with open("hexagonal_elevenlabs.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("Saved to hexagonal_elevenlabs.mp3")

ElevenLabs: 63156 bytes
Saved to hexagonal_elevenlabs.mp3


In [14]:
# OpenAI via TTSPort interface (same generate_speech function!)
text = "This demonstrates the hexagonal architecture pattern with OpenAI."

async with OpenAITTSAdapter() as adapter:
    config = OpenAIVoiceConfig(voice_id="nova")
    result = await generate_speech(adapter, text, config)
    
    print(f"OpenAI: {len(result.audio_bytes)} bytes")
    
    with open("hexagonal_openai.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("Saved to hexagonal_openai.mp3")

OpenAI: 83040 bytes
Saved to hexagonal_openai.mp3


## 6. Factory Pattern (Full Decoupling)

The examples above still couple to concrete adapters at instantiation. True hexagonal architecture uses a **factory** that hides adapter creation. Application code only imports the port interface.

In [1]:
"""
Factory module (would live in chatforge/factories/tts.py)
Application code imports ONLY this factory and TTSPort.
"""
from typing import Literal
from contextlib import asynccontextmanager

# Only the factory imports concrete adapters
from chatforge.adapters.tts import ElevenLabsTTSAdapter, OpenAITTSAdapter
from chatforge.ports.tts import TTSPort


# Default voice IDs per provider
DEFAULT_VOICES = {
    "elevenlabs": "21m00Tcm4TlvDq8ikWAM",  # Rachel
    "openai": "nova",
}


@asynccontextmanager
async def create_tts_adapter(
    provider: Literal["elevenlabs", "openai"] = "openai"
) -> TTSPort:
    """Factory that creates the appropriate TTS adapter."""
    if provider == "elevenlabs":
        adapter = ElevenLabsTTSAdapter()
    else:
        adapter = OpenAITTSAdapter()
    
    async with adapter:
        yield adapter

In [2]:
# Application code using factory - ElevenLabs
# Notice: Only imports from ports (interface), no concrete adapters!
from chatforge.ports.tts import VoiceConfig, AudioQuality

provider = "elevenlabs"
text = "This is fully decoupled using the factory pattern with ElevenLabs."

async with create_tts_adapter(provider) as tts:
    # Use base VoiceConfig - works with ANY provider!
    config = VoiceConfig(voice_id=DEFAULT_VOICES[provider])
    result = await tts.synthesize(text, config, quality=AudioQuality.STANDARD)
    
    print(f"Provider: {provider}")
    print(f"Bytes: {len(result.audio_bytes)}")
    
    with open("factory_elevenlabs.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("Saved to factory_elevenlabs.mp3")

Provider: elevenlabs
Bytes: 58559
Saved to factory_elevenlabs.mp3


In [3]:
# Application code using factory - OpenAI
# Same code structure, just change the provider string!

provider = "openai"
text = "This is fully decoupled using the factory pattern with OpenAI."

async with create_tts_adapter(provider) as tts:
    # Same base VoiceConfig works here too!
    config = VoiceConfig(voice_id=DEFAULT_VOICES[provider])
    result = await tts.synthesize(text, config, quality=AudioQuality.STANDARD)
    
    print(f"Provider: {provider}")
    print(f"Bytes: {len(result.audio_bytes)}")
    
    with open("factory_openai.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("Saved to factory_openai.mp3")

Provider: openai
Bytes: 75360
Saved to factory_openai.mp3


## 7. TTSService (Recommended for Apps)

The simplest way to use TTS in your app. Just import `TTSService` from services - no need to know about ports, adapters, or configs!

In [ ]:
# TTSService - ElevenLabs
# One import, simple API!
from chatforge.services import TTSService

async with TTSService("elevenlabs") as tts:
    result = await tts.generate(
        "Hello! This is the TTSService with ElevenLabs. Super simple!",
        quality="standard",
    )
    
    print(f"Provider: elevenlabs")
    print(f"Bytes: {len(result.audio_bytes)}")
    
    with open("service_elevenlabs.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("Saved to service_elevenlabs.mp3")

In [ ]:
# TTSService - OpenAI (default provider)
# Even simpler - just TTSService() uses OpenAI by default

async with TTSService() as tts:
    result = await tts.generate(
        "Hello! This is the TTSService with OpenAI. The simplest API!",
        quality="standard",
    )
    
    print(f"Provider: openai (default)")
    print(f"Bytes: {len(result.audio_bytes)}")
    
    with open("service_openai.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("Saved to service_openai.mp3")